<a href="https://colab.research.google.com/github/beena24/Research_works/blob/main/Deep_feat%2BLGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob
import cv2
import pandas as pd
from sklearn.model_selection import train_test_split
import os
from keras.applications.densenet import DenseNet169, DenseNet121
from keras.applications.mobilenet import MobileNet
from sklearn import preprocessing
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import classification_report
import lightgbm as lgb
import seaborn as sns

In [ ]:
SIZE = 224
images = []
images_labels = []

In [ ]:
ext = ['png', 'png', 'jpg']

for directory_path in glob.glob('/content/drive/MyDrive/chestx_ray8/*'):
    #print(directory_path)
    splited = directory_path.split("/")
    label = splited[-1]
    #print(splited)
    print(label)
    for e in ext:
      for img_path in glob.glob(os.path.join(directory_path,"*." + e)):
           #print(img_path)
           img = cv2.imread(img_path, cv2.IMREAD_COLOR)
           img = cv2.resize(img, (SIZE, SIZE))
           #images.append(img)
           np.append(img,images)
           #np.append(images,img)
           #np.append(images_labels,label)
           np.append(label,images_labels)


Pneumonia
Covid-19
No_findings


In [ ]:
print(img)

[[[ 17  17  17]
  [ 17  17  17]
  [ 18  18  18]
  ...
  [ 25  25  25]
  [ 34  34  34]
  [ 48  48  48]]

 [[ 17  17  17]
  [ 17  17  17]
  [ 18  18  18]
  ...
  [ 25  25  25]
  [ 34  34  34]
  [ 48  48  48]]

 [[ 17  17  17]
  [ 18  18  18]
  [ 17  17  17]
  ...
  [ 25  25  25]
  [ 34  34  34]
  [ 47  47  47]]

 ...

 [[212 212 212]
  [214 214 214]
  [217 217 217]
  ...
  [ 64  64  64]
  [ 67  67  67]
  [ 83  83  83]]

 [[220 220 220]
  [221 221 221]
  [226 226 226]
  ...
  [103 103 103]
  [ 88  88  88]
  [ 91  91  91]]

 [[229 229 229]
  [231 231 231]
  [234 234 234]
  ...
  [161 161 161]
  [145 145 145]
  [147 147 147]]]


In [ ]:

img = np.array(img)
labels = np.array(label)

In [ ]:
print(label)

No_findings


In [ ]:
img.shape

(224, 224, 3)

In [ ]:
np.unique(label)

array(['No_findings'], dtype='<U11')

In [ ]:
le = preprocessing.LabelEncoder()
#label=label.flatten
le.fit(label)
labels_encoded = le.transform(label)


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(img, labels_encoded, test_size=0.2, random_state=0)

In [ ]:
x_train, x_test = x_train / 255.0, x_test / 255.0


In [ ]:
dense_model = DenseNet169(include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')

In [ ]:
dense_features = dense_model.predict(x_train)

In [ ]:
features = dense_features.reshape(dense_features.shape[0], -1)


In [ ]:
features.shape

In [ ]:
model_mobile = MobileNet(weights='imagenet',include_top=False, input_shape=(SIZE, SIZE, 3), pooling='avg')

In [ ]:
mobile_features = model_mobile.predict(x_train)

In [ ]:
features_2 = mobile_features.reshape(mobile_features.shape[0], -1)


In [ ]:
features_2.shape

In [ ]:
combined_features = np.hstack((dense_features, mobile_features))

In [ ]:
params = {'learning_rate':0.24, 'n_iterations': 250, 'max_depth': 7, 'num_leaves': 105, 'n_estimators': 300, 'min_child_samples': 40 }


In [ ]:
lgb_classifier = lgb.LGBMClassifier(**params)

In [ ]:
lgb_classifier.fit(combined_features, y_train)


In [ ]:
X_test_dense_features = dense_model.predict(x_test)
X_test_dense_features = X_test_dense_features.reshape(X_test_dense_features.shape[0], -1)


In [ ]:
X_test_mobile_features = model_mobile.predict(x_test)
X_test_mobile_features = X_test_mobile_features.reshape(X_test_mobile_features.shape[0], -1)


In [ ]:
combined_test_features = np.hstack((X_test_dense_features, X_test_mobile_features))

In [ ]:
prediction = lgb_classifier.predict(combined_test_features)

In [ ]:
accuracy = accuracy_score(y_test, prediction)
accuracy


In [ ]:
cm = confusion_matrix(y_test, prediction)

In [ ]:
categories = ['Covid-19', 'No_findings', 'Pneumonia']
counts = ['{0:0.0f}'.format(value) for value in
          cm.flatten()]
group_percentages = ['{0:.2%}'.format(value) for value in
                      cm.flatten()/np.sum(cm)]
labels = [f'{v1}\n{v2}' for v1, v2 in
          zip(counts, group_percentages)]
labels = np.asarray(labels).reshape(3, 3)


sns_plot = sns.heatmap(cm, annot=labels, fmt='', cmap='Greys',xticklabels=categories, yticklabels=categories)

In [ ]:
def confusion_metrics(cm, class_name):
  report = classification_report(y_test, prediction, target_names=categories, output_dict=True)


  if class_name == 'Covid-19':
      sensitivity = cm[0,0]/sum(cm[0,:])
      specificity = (cm[1,1]+cm[2,2])/(cm[1,0]+cm[2,0]+cm[1,1]+cm[2,2])
      precision = report[class_name]['precision']
      f1 = report[class_name]['f1-score']
  elif class_name == 'No_findings':
      sensitivity = cm[1,1]/sum(cm[1,:])
      specificity = (cm[0,0]+cm[2,2])/(cm[0,1]+cm[2,1]+cm[0,0]+cm[2,2])
      precision = report[class_name]['precision']
      f1 = report[class_name]['f1-score']
  elif class_name == 'Pneumonia':
      sensitivity = cm[2,2]/sum(cm[2,:])
      specificity = (cm[0,0]+cm[1,1])/(cm[0,2]+cm[1,2]+cm[0,0]+cm[1,1])
      precision = report[class_name]['precision']
      f1 = report[class_name]['f1-score']


  result = {'Sensitivity': sensitivity*100, 'Specificity': specificity *
              100, 'Precision': precision*100, 'F1-Score': f1*100}
  return result


In [ ]:
performance = pd.DataFrame(
    columns=['Class', 'Sensitivity', 'Specificity', 'Precision', 'F1-Score', 'Accuracy'])
performance

In [ ]:
for c in categories:
  data = {**{'Class': f'{c}'}, **confusion_metrics(cm, c), **{'Accuracy': accuracy*100 }}
  performance = performance.append(data, ignore_index=True)


In [ ]:
performance